In [ ]:
import polars as pl
import altair as alt

DATA_DIR = r"C:\Users\ralex\Documents\GitHub\visualizacao-lab3\DADOS"

# O Altair limita 5.000 linhas por padrão.
# Nossos dados já estão agregados (uma linha por curso), então é seguro remover o limite.
alt.data_transformers.disable_max_rows()

## Visualização: NOTA_GER por Motivo de Escolha do Curso (QE_I25)

Vamos explorar como a nota geral (`NT_GER`) varia de acordo com o motivo que levou o estudante a escolher seu curso (`QE_I25`).

**Dados utilizados:**
- `microdados2023_arq3.txt` → `NT_GER` (nota geral)
- `microdados2023_arq31.txt` → `QE_I25` (motivo de escolha do curso)

---

### Por que Polars em vez de Pandas?

Polars é uma biblioteca de DataFrames escrita em Rust. Para arquivos grandes, ela usa bem menos RAM e é muito mais rápida. A sintaxe é parecida com Pandas, mas vamos ver as diferenças ao longo do notebook.

### Por que agregar antes de juntar?

Os arquivos não têm um identificador de estudante — só `NU_ANO` e `CO_CURSO`. Fazer join direto por `CO_CURSO` causaria um produto cartesiano (vários alunos por curso × vários alunos por curso = dezenas de milhões de linhas erradas).

A solução é **agregar primeiro por curso, juntar depois**:
1. `arq3` → média de `NT_GER` por `CO_CURSO`
2. `arq31` → motivo mais frequente (`QE_I25`) por `CO_CURSO`
3. Join 1:1 por `CO_CURSO` → resultado limpo, uma linha por curso

In [ ]:
arq1 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq1.csv",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "CO_GRUPO"],
    null_values=["", " "],
    infer_schema=False,
).with_columns([
    pl.col("CO_CURSO").cast(pl.Int64),
    pl.col("CO_GRUPO").cast(pl.Int64),
])

arq3 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq3.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "NT_GER"],
    null_values=["", " "],
    infer_schema=False,
).with_columns([
    pl.col("CO_CURSO").cast(pl.Int64),
    pl.col("NT_GER").str.replace(",", ".").cast(pl.Float64, strict=False),
])

arq31 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq31.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "QE_I25"],
    null_values=["", " "],
)

# Passo 1: nota média por curso (excluindo ausentes)
media_por_curso = (
    arq3
    .filter(pl.col("NT_GER").is_not_null() & (pl.col("NT_GER") > 0))
    .group_by("CO_CURSO")
    .agg(pl.col("NT_GER").mean().alias("NT_GER_media"))
)

# Passo 2: associa CO_GRUPO a cada curso (um curso → um grupo)
# arq1 tem múltiplas linhas por CO_CURSO (um por aluno), pegamos distinct
grupo_por_curso = arq1.unique(subset=["CO_CURSO"])

media_com_grupo = media_por_curso.join(grupo_por_curso, on="CO_CURSO", how="inner")

# Passo 3: nota média por CO_GRUPO (a "dificuldade" de cada tipo de prova)
media_por_grupo = (
    media_com_grupo
    .group_by("CO_GRUPO")
    .agg(pl.col("NT_GER_media").mean().alias("NT_GER_media_grupo"))
)

# Passo 4: normaliza — nota do curso dividida pela média do seu grupo
# valor > 1: curso acima da média do grupo | valor < 1: abaixo
media_com_grupo = (
    media_com_grupo
    .join(media_por_grupo, on="CO_GRUPO", how="inner")
    .with_columns(
        (pl.col("NT_GER_media") / pl.col("NT_GER_media_grupo")).alias("NT_GER_norm")
    )
)

# Passo 5: quantos alunos de cada curso reportaram cada motivo
contagem_motivos = (
    arq31
    .filter(pl.col("QE_I25").is_not_null())
    .group_by(["CO_CURSO", "QE_I25"])
    .agg(pl.len().alias("n_alunos"))
)

# Passo 6: junta tudo — (curso × motivo) com nota normalizada
df = contagem_motivos.join(media_com_grupo, on="CO_CURSO", how="inner")

motivos = {
    "A": "Mercado de trabalho",
    "B": "Influência familiar",
    "C": "Valorização profissional",
    "D": "Prestígio social",
    "E": "Vocação",
    "F": "Modalidade a distância",
    "G": "Baixa concorrência",
    "H": "Outro motivo",
}
df = df.with_columns(pl.col("QE_I25").replace(motivos).alias("Motivo"))

print(f"Linhas (curso × motivo): {df.shape[0]:,}")
print(f"Nulos em NT_GER_norm: {df['NT_GER_norm'].null_count()}")
df.head()

In [ ]:
# Verifica se CO_CURSO é chave única
print(f"arq3  — total: {arq3.shape[0]:,} | CO_CURSO únicos: {arq3['CO_CURSO'].n_unique():,}")
print(f"arq31 — total: {arq31.shape[0]:,} | CO_CURSO únicos: {arq31['CO_CURSO'].n_unique():,}")
print()
print("Top 5 cursos com mais linhas em arq3:")
print(
    arq3.group_by("CO_CURSO")
    .agg(pl.len().alias("n_alunos"))
    .sort("n_alunos", descending=True)
    .head(5)
)

In [ ]:
import numpy as np

_df = (
    df.select(["NT_GER_norm", "Motivo", "n_alunos"])
    .filter(pl.col("NT_GER_norm").is_not_null())
    # Arredonda para 2 casas decimais e agrega — pontos com diferença no 3º/4º decimal viram um só
    .with_columns(pl.col("NT_GER_norm").round(2))
    .group_by(["Motivo", "NT_GER_norm"])
    .agg(pl.col("n_alunos").sum())
    .to_pandas()
)

print(f"Pontos após agrupamento: {len(_df):,}")

ordem = (
    _df.groupby("Motivo")
    .apply(lambda g: np.average(g["NT_GER_norm"], weights=g["n_alunos"]), include_groups=False)
    .sort_values(ascending=False)
    .index.tolist()
)

if "Outro motivo" in ordem:
    ordem.remove("Outro motivo")
    ordem.append("Outro motivo")

rng = np.random.default_rng(42)
_df["jitter"] = rng.uniform(-0.3, 0.3, size=len(_df))

chart = (
    alt.Chart(_df)
    .mark_point(opacity=0.4, filled=True)
    .encode(
        x=alt.X("Motivo:N", sort=ordem, title="Motivo de escolha do curso"),
        xOffset=alt.XOffset("jitter:Q", scale=alt.Scale(domain=[-0.5, 0.5])),
        y=alt.Y("NT_GER_norm:Q",
                title="Nota relativa ao grupo (1 = média do grupo)",
                scale=alt.Scale(zero=False)),
        size=alt.Size("n_alunos:Q", title="Nº de alunos",
                      scale=alt.Scale(range=[10, 300])),
        color=alt.Color("Motivo:N", legend=None),
        tooltip=["Motivo", "NT_GER_norm", "n_alunos"],
    )
    .properties(
        title="Nota relativa ao CO_GRUPO × Motivo de Escolha (QE_I25)",
        width=650,
        height=420,
    )
)

chart

In [ ]:
_df2 = (
    df.select(["NT_GER_media", "Motivo", "n_alunos"])
    .filter(pl.col("NT_GER_media").is_not_null())
    .with_columns(pl.col("NT_GER_media").round(2))
    .group_by(["Motivo", "NT_GER_media"])
    .agg(pl.col("n_alunos").sum())
    .to_pandas()
)

print(f"Pontos: {len(_df2):,}")

ordem2 = (
    _df2.groupby("Motivo")
    .apply(lambda g: np.average(g["NT_GER_media"], weights=g["n_alunos"]), include_groups=False)
    .sort_values(ascending=False)
    .index.tolist()
)

if "Outro motivo" in ordem2:
    ordem2.remove("Outro motivo")
    ordem2.append("Outro motivo")

rng2 = np.random.default_rng(42)
_df2["jitter"] = rng2.uniform(-0.3, 0.3, size=len(_df2))

chart2 = (
    alt.Chart(_df2)
    .mark_point(opacity=0.4, filled=True)
    .encode(
        x=alt.X("Motivo:N", sort=ordem2, title="Motivo de escolha do curso"),
        xOffset=alt.XOffset("jitter:Q", scale=alt.Scale(domain=[-0.5, 0.5])),
        y=alt.Y("NT_GER_media:Q",
                title="Nota média do curso",
                scale=alt.Scale(zero=False)),
        size=alt.Size("n_alunos:Q", title="Nº de alunos",
                      scale=alt.Scale(range=[10, 300])),
        color=alt.Color("Motivo:N", legend=None),
        tooltip=["Motivo", "NT_GER_media", "n_alunos"],
    )
    .properties(
        title="Nota média do curso × Motivo de Escolha (QE_I25) — sem normalização por CO_GRUPO",
        width=650,
        height=420,
    )
)

chart2

---
## Versão violino

`transform_density` calcula a distribuição de densidade (KDE) diretamente no Altair, sem precisar de biblioteca extra. `stack='center'` espelha a curva dos dois lados, criando a forma de violino. Usamos `column` para criar um painel por motivo.

In [ ]:
# Violino — nota normalizada por CO_GRUPO
violin_norm = (
    alt.Chart(_df)
    .transform_density(
        "NT_GER_norm",
        as_=["NT_GER_norm", "density"],
        groupby=["Motivo"],
    )
    .mark_area(orient="horizontal")
    .encode(
        y=alt.Y("NT_GER_norm:Q", title="Nota relativa ao grupo (1 = média)"),
        x=alt.X(
            "density:Q",
            stack="center",
            title=None,
            axis=alt.Axis(labels=False, ticks=False, grid=False, domain=False),
        ),
        color=alt.Color("Motivo:N", legend=None),
        column=alt.Column(
            "Motivo:N",
            sort=ordem,
            header=alt.Header(labelOrient="bottom", title=None, labelAngle=-30),
        ),
    )
    .properties(width=80, height=380,
                title="Nota relativa ao CO_GRUPO × Motivo (QE_I25)")
)

violin_norm

In [ ]:
# Violino — nota bruta (sem normalização por CO_GRUPO)
violin_bruto = (
    alt.Chart(_df2)
    .transform_density(
        "NT_GER_media",
        as_=["NT_GER_media", "density"],
        groupby=["Motivo"],
    )
    .mark_area(orient="horizontal")
    .encode(
        y=alt.Y("NT_GER_media:Q", title="Nota média do curso"),
        x=alt.X(
            "density:Q",
            stack="center",
            title=None,
            axis=alt.Axis(labels=False, ticks=False, grid=False, domain=False),
        ),
        color=alt.Color("Motivo:N", legend=None),
        column=alt.Column(
            "Motivo:N",
            sort=ordem2,
            header=alt.Header(labelOrient="bottom", title=None, labelAngle=-30),
        ),
    )
    .properties(width=80, height=380,
                title="Nota média do curso × Motivo (QE_I25) — sem normalização")
)

violin_bruto

In [ ]:
# Diagnóstico: verificar o que está no df antes de plotar
print("Shape:", df.shape)
print("\nColunas:", df.columns)
print("\nValores únicos de QE_I25:", df["QE_I25"].unique().to_list())
print("Valores únicos de Motivo:", df["Motivo"].unique().to_list())
print("\nNulos em NT_GER_media:", df["NT_GER_media"].null_count())
print("Nulos em Motivo:", df["Motivo"].null_count())
print()
df.head(10)